# COMP 469 Lab 02: Solving Problems by Searching -- INSTRUCTOR SOLUTIONS

**Instructor copy** | **Week 2** | **AIMA Chapter 3** | **3-hour lab**

This notebook is the fully solved version of `COMP469_Lab02_Search.ipynb`, for instructor use only. Do not distribute this file to students.

It contains:

- All TODOs completed with working code.
- Expected console output inline as comments so you can sanity-check a student's run without re-executing everything.
- An **Instructor Answer Key** callout after every checkpoint, with a model answer and the key idea each question is trying to surface.
- Teaching notes flagging where students commonly get stuck.

A companion file, `COMP469_Lab02_Instructor_Notes.md`, has the section-by-section pacing, grading rubric, and common-mistakes list. This notebook is the code reference; that file is the classroom-management reference.


## 0. Python `.env` Setup

Same as the student copy. If your lab room already has the `comp469` kernel from Lab 01, students can skip straight to Section 1.

**Instructor note:** the only new package requirement versus Lab 01 is `networkx` (for `show_map`). If your lab image was built for Lab 01 only, push an update or have students run:

```bash
pip install networkx ipywidgets
```

before the setup cell, or the import in Section 1 will fail.


## 1. Setup Cell

Run this cell first.


In [ ]:
import os, sys

_root = os.path.abspath(os.getcwd())
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, 'aima')):
    _root = os.path.dirname(_root)

if not os.path.isdir(os.path.join(_root, 'aima')):
    raise FileNotFoundError('Could not find the bundled aima/ support package')

if _root not in sys.path:
    sys.path.insert(0, _root)

import random
import time
from collections import deque

import matplotlib.pyplot as plt
import networkx as nx

from aima.search import Problem, Node, GraphProblem, UndirectedGraph, memoize, PriorityQueue, infinity
from aima.notebook_utils import psource, show_map

import warnings
warnings.filterwarnings("ignore")

random.seed(469)
print("Setup complete. You are ready for Lab 02.")
# Expected output: Setup complete. You are ready for Lab 02.


## 2. Problem Formulation Review

### PEAS-style problem formulation -- SOLUTION

| Component | Answer |
|---|---|
| State space | The set of Romanian cities reachable via the road network in `romania_map` (20 cities) |
| Initial state | Arad |
| Goal test | `state == "Bucharest"` |
| Actions | `Go(city)` for each city directly connected to the current city by a road |
| Action cost | The road distance (km) between the two cities, as given by the edge weight |

### Classification -- SOLUTION

| Property | Choice | Justification |
|---|---|---|
| Fully or partially observable? | Fully observable | The agent always knows exactly which city it is in and the full map is known in advance |
| Single-agent or multi-agent? | Single-agent | Only one agent is moving through the environment; no other agents affect outcomes |
| Deterministic or stochastic? | Deterministic | Taking the road to a given city always results in arriving at that city |
| Episodic or sequential? | Sequential | Each move affects which moves are available/needed next; this is not a series of independent decisions |
| Static or dynamic? | Static | The map does not change while the agent is deliberating |
| Discrete or continuous? | Discrete | There is a finite, countable set of cities and roads |

**Teaching note:** students sometimes answer "partially observable" because the agent does not know the map before Chapter 3's framing gives it one. Clarify that in this chapter's formulation, the agent already has the map (Figure 3.1) -- observability is about the agent's ability to perceive *its own current state*, not about how the agent originally acquired its knowledge of the environment.

Run the next two cells to see how AIMA defines the abstract `Problem` class and the `Node` class.


In [ ]:
psource(Problem)


**Instructor Answer Key:** `goal_test(state)` generalizes beyond a single fixed goal because some problems have multiple acceptable goal states, or a goal defined by a *property* rather than a specific state (e.g., "no dirt in any location" in the vacuum world, or "every queen placed with no conflicts" in N-Queens). A single `goal` attribute cannot express "any state satisfying property P," but a `goal_test` method can. Accept any example where the goal is a condition rather than a single named state (8-puzzle also works if a student frames it as "any state matching the tile pattern," though in this lab we do use one fixed goal state for the 8-puzzle).


In [ ]:
psource(Node)


**Instructor Answer Key:** The state space graph only needs to describe *which states exist and how they connect* -- it has no notion of "how did we get here." The search tree, on the other hand, is built by the search algorithm to keep track of the specific path taken to reach a state, so that once a goal node is found, the algorithm can reconstruct the sequence of actions (the solution) by walking back up `parent` pointers. Without `parent`, `action`, and `path_cost` on each node, the algorithm would find *that* a goal is reachable but not *how* to reach it, and could not compare the cost of different paths to the same state.


## 3. Define the Romania Map and the Route-Finding Problem


In [ ]:
romania_map = UndirectedGraph(dict(
    Arad=dict(Zerind=75, Sibiu=140, Timisoara=118),
    Bucharest=dict(Urziceni=85, Pitesti=101, Giurgiu=90, Fagaras=211),
    Craiova=dict(Drobeta=120, Rimnicu=146, Pitesti=138),
    Drobeta=dict(Mehadia=75),
    Eforie=dict(Hirsova=86),
    Fagaras=dict(Sibiu=99),
    Hirsova=dict(Urziceni=98),
    Iasi=dict(Vaslui=92, Neamt=87),
    Lugoj=dict(Timisoara=111, Mehadia=70),
    Oradea=dict(Zerind=71, Sibiu=151),
    Pitesti=dict(Rimnicu=97),
    Rimnicu=dict(Sibiu=80),
    Urziceni=dict(Vaslui=142),
))
romania_map.locations = dict(
    Arad=(91, 492), Bucharest=(400, 327), Craiova=(253, 288), Drobeta=(165, 299),
    Eforie=(562, 293), Fagaras=(305, 449), Giurgiu=(375, 270), Hirsova=(534, 350),
    Iasi=(473, 506), Lugoj=(165, 379), Mehadia=(168, 339), Neamt=(406, 537),
    Oradea=(131, 571), Pitesti=(320, 368), Rimnicu=(233, 410), Sibiu=(207, 457),
    Timisoara=(94, 410), Urziceni=(456, 350), Vaslui=(509, 444), Zerind=(108, 531),
)

# SOLUTION 1: the agent starts in Arad and must reach Bucharest.
romania_problem = GraphProblem("Arad", "Bucharest", romania_map)

print("Initial state:", romania_problem.initial)
print("Is Bucharest the goal?", romania_problem.goal_test("Bucharest"))
# Expected output:
# Initial state: Arad
# Is Bucharest the goal? True


## 4. Visualize the Map


In [ ]:
node_colors = {node: 'white' for node in romania_map.locations.keys()}
node_positions = romania_map.locations
node_label_pos = {k: [v[0], v[1] - 10] for k, v in romania_map.locations.items()}
edge_weights = {(k, k2): v2 for k, v in romania_map.graph_dict.items() for k2, v2 in v.items()}

romania_graph_data = {
    'graph_dict': romania_map.graph_dict,
    'node_colors': node_colors,
    'node_positions': node_positions,
    'node_label_positions': node_label_pos,
    'edge_weights': edge_weights,
}

show_map(romania_graph_data)


### Checkpoint 1 -- Instructor Answer Key

1. **State space size:** The state space is the 20 cities in `romania_map` (Arad, Bucharest, Craiova, Drobeta, Eforie, Fagaras, Giurgiu, Hirsova, Iasi, Lugoj, Mehadia, Neamt, Oradea, Pitesti, Rimnicu, Sibiu, Timisoara, Urziceni, Vaslui, Zerind). Students can verify the count with `len(romania_map.nodes())`.
2. **Edge weights:** They represent road distance in kilometers between two directly connected cities, and are summed along a path to compute path cost / `g(n)`.
3. **Tree vs. graph search:** Graph search is preferred, because the map contains cycles -- for example Arad-Sibiu-Rimnicu-Pitesti-Craiova-Drobeta-Mehadia-Lugoj-Timisoara-Arad is a cycle. Tree search (which does not track an explored set) would re-expand the same cities repeatedly and could loop forever on a graph with cycles; graph search's `explored` set prevents this.
4. **Why locations are needed:** The (x, y) coordinates are not used by uninformed search or by path cost at all -- they exist purely to compute the **heuristic** (straight-line distance) used later by greedy best-first search and A*, and to lay out the visualization.

**Teaching note:** watch for students who answer "20 states" without being able to name where that number comes from -- have them actually run `len(romania_map.nodes())` rather than counting from the picture, since it's easy to miscount from `show_map`'s layout.


## 5. Breadth-First Graph Search (Uninformed)


In [ ]:
def breadth_first_graph_search(problem):
    """Search the shallowest nodes in the search tree first. Returns (goal_node, nodes_expanded)
    or (None, nodes_expanded) if no solution exists."""
    nodes_expanded = 0

    node = Node(problem.initial)
    if problem.goal_test(node.state):
        return node, nodes_expanded

    frontier = deque([node])
    explored = set()

    while frontier:
        # SOLUTION 2: FIFO pop from the left.
        node = frontier.popleft()

        explored.add(node.state)
        nodes_expanded += 1

        for child in node.expand(problem):
            # SOLUTION 3: only consider children that are neither explored nor already frontier.
            if child.state not in explored and child not in frontier:
                if problem.goal_test(child.state):
                    return child, nodes_expanded
                frontier.append(child)

    return None, nodes_expanded


bfs_solution, bfs_expanded = breadth_first_graph_search(romania_problem)
print("Path:", bfs_solution.solution())
print("Path cost:", bfs_solution.path_cost)
print("Nodes expanded:", bfs_expanded)
# Expected output:
# Path: ['Sibiu', 'Fagaras', 'Bucharest']
# Path cost: 450
# Nodes expanded: 6


**Teaching note:** the most common bug here is students writing `frontier.pop()` (which pops from the *right* of a `deque`) instead of `frontier.popleft()`. The symptom is that BFS silently behaves like DFS -- it will still return *a* path, so students may not notice unless they compare the path/cost/nodes-expanded against the expected output above. Have them print `type(frontier)` and remind them `deque.pop()` and `deque.popleft()` are different ends.


## 6. Depth-First Graph Search (Uninformed)


In [ ]:
def depth_first_graph_search(problem):
    """Search the deepest nodes in the search tree first. Returns (goal_node, nodes_expanded)
    or (None, nodes_expanded) if no solution exists."""
    nodes_expanded = 0

    node = Node(problem.initial)
    if problem.goal_test(node.state):
        return node, nodes_expanded

    frontier = [node]
    explored = set()

    while frontier:
        # SOLUTION 4: LIFO pop from the end of the list.
        node = frontier.pop()

        explored.add(node.state)
        nodes_expanded += 1

        for child in node.expand(problem):
            if child.state not in explored and child not in frontier:
                if problem.goal_test(child.state):
                    return child, nodes_expanded
                # SOLUTION 5: push the child onto the stack.
                frontier.append(child)

    return None, nodes_expanded


dfs_solution, dfs_expanded = depth_first_graph_search(romania_problem)
print("Path:", dfs_solution.solution())
print("Path cost:", dfs_solution.path_cost)
print("Nodes expanded:", dfs_expanded)
# Expected output (order of neighbors as stored in the dict determines the exact path;
# with the graph exactly as given above, using Python 3.7+ dict ordering):
# Path: ['Timisoara', 'Lugoj', 'Mehadia', 'Drobeta', 'Craiova', 'Pitesti', 'Bucharest']
# Path cost: 733
# Nodes expanded: 7


**Teaching note:** DFS's exact path depends on the order neighbors are stored in the dictionary passed to `UndirectedGraph`, and on `make_undirected` (which adds reverse edges). If a student's DFS path differs from the expected output above but their path cost is a valid path in the map and their nodes-expanded count is reasonable, do not mark it wrong -- confirm their algorithm structure matches the BFS-to-DFS diff (only the pop end changed) rather than requiring an exact string match to the expected path.

### Checkpoint 2 -- Instructor Answer Key

1. **Same path?** No. BFS returns `Sibiu -> Fagaras -> Bucharest` (cost 450); DFS returns a much longer route through Timisoara/Lugoj/Mehadia/Drobeta/Craiova/Pitesti (cost 733). Different paths, different (and much higher) cost for DFS.
2. **Nodes expanded:** DFS expanded slightly more nodes on this instance (7 vs. 6) -- but this is coincidental to this particular map/goal, not a general rule. On a wide, shallow graph DFS can expand far more nodes than BFS; on a deep, narrow graph the reverse can happen. The key point for discussion is that node count alone doesn't tell the full story -- path cost matters too.
3. **Optimality:** Neither BFS nor DFS uses edge weights when choosing what to expand, so neither is guaranteed to find the lowest-cost path. BFS only guarantees the *fewest edges* (shallowest node), which is only the same as lowest cost when all action costs are equal. DFS gives no optimality guarantee of any kind.
4. **When would BFS be optimal?** BFS is optimal whenever every action has the same cost (e.g., unit-cost problems like the 8-puzzle, where every move costs 1). On the Romania map, road distances vary, so BFS's guarantee does not apply here.


## 7. Uniform-Cost Search (Cost-Aware)


In [ ]:
def uniform_cost_search(problem):
    """Best-first graph search with f(n) = g(n) = path cost. Returns (goal_node, nodes_expanded)
    or (None, nodes_expanded) if no solution exists."""
    nodes_expanded = 0

    # SOLUTION 6: f is the path cost of a node.
    f = memoize(lambda n: n.path_cost, 'f')

    node = Node(problem.initial)
    frontier = PriorityQueue('min', f)
    frontier.append(node)
    explored = set()

    while frontier:
        node = frontier.pop()

        if problem.goal_test(node.state):
            return node, nodes_expanded

        explored.add(node.state)
        nodes_expanded += 1

        for child in node.expand(problem):
            if child.state not in explored and child not in frontier:
                frontier.append(child)
            elif child in frontier:
                incumbent = frontier[child]
                # SOLUTION 7: replace the incumbent if the new path is cheaper.
                if f(child) < incumbent:
                    del frontier[child]
                    frontier.append(child)

    return None, nodes_expanded


ucs_solution, ucs_expanded = uniform_cost_search(romania_problem)
print("Path:", ucs_solution.solution())
print("Path cost:", ucs_solution.path_cost)
print("Nodes expanded:", ucs_expanded)
# Expected output:
# Path: ['Sibiu', 'Rimnicu', 'Pitesti', 'Bucharest']
# Path cost: 418
# Nodes expanded: 12


**Teaching note:** this is the point in the lab where students should notice UCS finds a *cheaper* path (418) than BFS (450) even though it expands more nodes (12 vs. 6). If a student's UCS returns path cost 450 identical to BFS, they likely left `f` returning a constant (e.g., forgot to replace `None`/`0.0`) so the priority queue is not actually ordering by cost -- have them print `f(node)` for a couple of nodes to check it is *not* constant.

Also common: forgetting the `goal_test` check happens **after popping from the frontier** (not when a child is generated), which is required for uniform-cost search to guarantee optimality. If a student copies the BFS/DFS pattern of testing the goal at child-generation time, UCS may return a suboptimal path on graphs where a cheaper path to the goal is discovered later. The scaffold above already places the goal test correctly after `pop()`; if students restructure this cell heavily, this is worth flagging.


## 8. A Heuristic: Straight-Line Distance


In [ ]:
import math

def straight_line_distance(state, goal, locations):
    x1, y1 = locations[state]
    x2, y2 = locations[goal]

    # SOLUTION 8: Euclidean distance.
    return math.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)


def romania_h(node):
    return straight_line_distance(node.state, romania_problem.goal, romania_map.locations)


print("h(Bucharest) =", straight_line_distance("Bucharest", "Bucharest", romania_map.locations))
print("h(Arad) =", romania_h(Node("Arad")))
# Expected output:
# h(Bucharest) = 0.0
# h(Arad) = approximately 350.3, computed from the (x, y) coordinates given above for Arad
# and Bucharest. (If your aima/ package ships its own romania_map.locations table with
# different coordinates than the ones hardcoded in this lab, the exact number will differ --
# what matters is that it is a positive number smaller than the actual road distance, which
# is the point of Checkpoint 1, question 4 and the admissibility discussion in Section 8.)


## 9. Greedy Best-First Search and A* Search (Informed)


In [ ]:
def best_first_graph_search(problem, f):
    """Search nodes with the lowest f(n) first. Returns (goal_node, nodes_expanded)
    or (None, nodes_expanded) if no solution exists."""
    nodes_expanded = 0
    f = memoize(f, 'f')

    node = Node(problem.initial)
    frontier = PriorityQueue('min', f)
    frontier.append(node)
    explored = set()

    while frontier:
        node = frontier.pop()

        if problem.goal_test(node.state):
            return node, nodes_expanded

        explored.add(node.state)
        nodes_expanded += 1

        for child in node.expand(problem):
            if child.state not in explored and child not in frontier:
                frontier.append(child)
            elif child in frontier:
                incumbent = frontier[child]
                if f(child) < incumbent:
                    del frontier[child]
                    frontier.append(child)

    return None, nodes_expanded


def greedy_best_first_search(problem, h):
    # SOLUTION 9: f(n) = h(n).
    return best_first_graph_search(problem, lambda n: h(n))


def astar_search(problem, h):
    # SOLUTION 10: f(n) = g(n) + h(n).
    return best_first_graph_search(problem, lambda n: n.path_cost + h(n))


greedy_solution, greedy_expanded = greedy_best_first_search(romania_problem, romania_h)
print("Greedy path:", greedy_solution.solution())
print("Greedy path cost:", greedy_solution.path_cost)
print("Greedy nodes expanded:", greedy_expanded)
print()

astar_solution, astar_expanded = astar_search(romania_problem, romania_h)
print("A* path:", astar_solution.solution())
print("A* path cost:", astar_solution.path_cost)
print("A* nodes expanded:", astar_expanded)
# Expected output:
# Greedy path: ['Sibiu', 'Fagaras', 'Bucharest']
# Greedy path cost: 450
# Greedy nodes expanded: 3
#
# A* path: ['Sibiu', 'Rimnicu', 'Pitesti', 'Bucharest']
# A* path cost: 418
# A* nodes expanded: 5


### Checkpoint 3 -- Instructor Answer Key

1. **Greedy vs. A*:** Greedy returns the same 450-cost path as BFS (`Sibiu -> Fagaras -> Bucharest`), while A* matches UCS's optimal 418-cost path (`Sibiu -> Rimnicu -> Pitesti -> Bucharest`). Greedy is more expensive because it only looks at `h(n)` (estimated distance to Bucharest) and ignores the cost already paid to get there -- Fagaras looks attractive because it's geographically close to Bucharest, even though the Rimnicu/Pitesti route is cheaper overall.
2. **Nodes expanded ranking (fewest to most) on this instance:** Greedy (3) < A* (5) < BFS (6) < DFS (7) < UCS (12). Greedy expands the fewest because it beelines toward the goal using only the heuristic; UCS expands the most among the "good" algorithms because it has no information about *direction*, only cost-so-far, so it must explore in all directions equally until it is sure it has the cheapest path.
3. **Overestimating heuristic:** If `h` could overestimate the true remaining distance, A* could be misled into preferring a path that only *looks* cheaper due to an inflated `f(n)` on the competing branch, and could return a suboptimal solution -- the optimality guarantee for A* depends specifically on `h` never overestimating (admissibility).
4. **Completeness vs. optimality:** Greedy best-first search is complete on this finite graph (it will eventually find a path to the goal if one exists, since the graph is finite and it maintains an explored set), but it is not optimal, as shown by its 450-cost result versus A*'s 418. Completeness is about *whether* a solution is found; optimality is about whether the *best* solution is found.

**Teaching note:** a subtlety worth surfacing in discussion -- greedy best-first search is not complete in general for infinite state spaces or when loops are handled poorly, but with a graph-search explored set on a finite graph like Romania, it is complete here. Encourage students to distinguish "complete on this problem" from "complete in general," which is a distinction AIMA emphasizes repeatedly for different algorithms.


## 10. The 8-Puzzle and Heuristic Design


In [ ]:
goal_state = (1, 2, 3, 4, 5, 6, 7, 8, 0)

index_goal = {1: (0, 0), 2: (0, 1), 3: (0, 2),
              4: (1, 0), 5: (1, 1), 6: (1, 2),
              7: (2, 0), 8: (2, 1), 0: (2, 2)}


def misplaced_tiles(node):
    state = node.state
    # SOLUTION 11: count mismatches, excluding the blank tile (0).
    count = 0
    for i in range(9):
        if state[i] != goal_state[i] and state[i] != 0:
            count += 1
    return count


def manhattan_distance(node):
    state = node.state
    total = 0
    for i in range(9):
        tile = state[i]
        if tile == 0:
            continue
        current_row, current_col = i // 3, i % 3
        goal_row, goal_col = index_goal[tile]
        # SOLUTION 12: Manhattan distance for this tile.
        total += abs(current_row - goal_row) + abs(current_col - goal_col)
    return total


test_state = (2, 4, 3, 1, 5, 6, 7, 8, 0)
test_node = Node(test_state)
print("Misplaced tiles:", misplaced_tiles(test_node))
print("Manhattan distance:", manhattan_distance(test_node))
# Expected output:
# Misplaced tiles: 3
# Manhattan distance: 4


In [ ]:
class EightPuzzleProblem(Problem):
    """A minimal 8-puzzle Problem, independent of any pre-built AIMA puzzle class."""

    def __init__(self, initial, goal=goal_state):
        super().__init__(initial, goal)

    def actions(self, state):
        blank = state.index(0)
        row, col = blank // 3, blank % 3
        possible = []
        if row > 0:
            possible.append("Up")
        if row < 2:
            possible.append("Down")
        if col > 0:
            possible.append("Left")
        if col < 2:
            possible.append("Right")
        return possible

    def result(self, state, action):
        blank = state.index(0)
        row, col = blank // 3, blank % 3
        delta = {"Up": -3, "Down": 3, "Left": -1, "Right": 1}
        swap_with = blank + delta[action]
        state = list(state)
        state[blank], state[swap_with] = state[swap_with], state[blank]
        return tuple(state)

    def goal_test(self, state):
        return state == self.goal

    def path_cost(self, c, state1, action, state2):
        return c + 1


puzzle = EightPuzzleProblem((2, 4, 3, 1, 5, 6, 7, 8, 0))

for name, h in [("misplaced_tiles", misplaced_tiles), ("manhattan_distance", manhattan_distance)]:
    start = time.perf_counter()
    solution, expanded = astar_search(puzzle, h)
    elapsed = time.perf_counter() - start
    print(f"{name}: moves={len(solution.solution())}, nodes_expanded={expanded}, time={elapsed:.4f}s")
# Expected output (exact nodes_expanded may vary slightly by Python version / tie-breaking
# order in the priority queue, but moves should always be 8, and manhattan_distance should
# expand fewer or equal nodes than misplaced_tiles):
# misplaced_tiles: moves=8, nodes_expanded=12, time=0.000Xs
# manhattan_distance: moves=8, nodes_expanded=10, time=0.000Xs


### Checkpoint 4 -- Instructor Answer Key

1. **Same move count:** Both heuristics are admissible (neither overestimates true remaining cost), and A* with an admissible heuristic is guaranteed to return an *optimal* solution regardless of which admissible heuristic is used. Since there is only one optimal move count for a given start/goal pair, both runs must report the same number of moves (8 in the example instance), even though the *number of nodes expanded* along the way differs.
2. **More informed heuristic:** `manhattan_distance` expanded fewer nodes (10 vs. 12 in the example run), so it is the more informed heuristic -- it gives a tighter (closer to true cost) estimate than simply counting misplaced tiles, so A* can prune more of the search space.
3. **Including the blank:** Including the blank tile in the misplaced-tiles count would still be admissible in principle only if moving the blank to its goal position is guaranteed to cost at least one move whenever it's out of place -- which is true here, so including it would still not overestimate. However, the conventional formulation excludes the blank because the blank is not a "tile" being placed by the puzzle's goal condition, and including it can occasionally make the heuristic slightly less informative in some formulations. Accept either answer (include or exclude) as long as the student's reasoning about admissibility is correct; the important point is that admissibility only requires "never overestimate," not "count every square."
4. **Why the relaxation cannot overestimate:** Manhattan distance for a single tile is the number of moves that tile would need if it could travel directly to its goal position ignoring every other tile. In the real puzzle, other tiles can only ever force *additional* moves (by blocking a direct path or requiring detours) -- they can never make it possible to move a tile to its goal in *fewer* moves than the straight Manhattan path. Summing this per-tile lower bound therefore produces a total that is also a lower bound (or exact match) on the true cost, so it can never overestimate.


## 11. Controlled Experiment


In [ ]:
route_pairs = [
    ("Arad", "Bucharest"),
    ("Oradea", "Eforie"),
    ("Timisoara", "Vaslui"),
    ("Neamt", "Craiova"),
    ("Zerind", "Hirsova"),
]

algorithms = {
    "BFS": lambda problem: breadth_first_graph_search(problem),
    "DFS": lambda problem: depth_first_graph_search(problem),
    "UCS": lambda problem: uniform_cost_search(problem),
    "Greedy": lambda problem: greedy_best_first_search(
        problem, lambda n: straight_line_distance(n.state, problem.goal, romania_map.locations)
    ),
    "A*": lambda problem: astar_search(
        problem, lambda n: straight_line_distance(n.state, problem.goal, romania_map.locations)
    ),
}


def run_experiment(route_pairs, algorithms):
    rows = []

    for start, goal in route_pairs:
        problem = GraphProblem(start, goal, romania_map)

        for algo_name, run_algo in algorithms.items():
            # SOLUTION 13: time the algorithm call.
            start_time = time.perf_counter()
            solution, nodes_expanded = run_algo(problem)
            end_time = time.perf_counter()
            elapsed = end_time - start_time

            rows.append({
                "algorithm": algo_name,
                "start": start,
                "goal": goal,
                "path_cost": solution.path_cost if solution else None,
                "nodes_expanded": nodes_expanded,
                "time_seconds": elapsed,
            })

    return rows


results = run_experiment(route_pairs, algorithms)
results[:5]


**Teaching note:** two of the five routes (Neamt to Craiova, in particular) require crossing the whole map and may take DFS noticeably longer / expand many more nodes than the others, since DFS has no guidance and this graph has several long dead-end branches (e.g., Iasi/Neamt and Eforie/Hirsova/Urziceni). If a student's DFS run for that pair looks unusually slow, that is expected behavior, not a bug -- it's actually a good discussion point for Checkpoint 5.

If any (start, goal) pair has no path at all (it shouldn't, with this map, since it's fully connected), `run_algo` would return `(None, nodes_expanded)`; the code above guards against a crash on `solution.path_cost` with the `if solution else None` conditional, so this is safe even if students test with custom disconnected graphs during the extension challenge.


## 12. Summarize and Plot Your Results


In [ ]:
def summarize_results(results):
    summary = {}
    algo_names = sorted({row["algorithm"] for row in results})

    for algo_name in algo_names:
        rows = [row for row in results if row["algorithm"] == algo_name]
        summary[algo_name] = {
            "avg_nodes_expanded": sum(row["nodes_expanded"] for row in rows) / len(rows),
            "avg_path_cost": sum(row["path_cost"] for row in rows) / len(rows),
            "avg_time_seconds": sum(row["time_seconds"] for row in rows) / len(rows),
        }

    return summary


summary = summarize_results(results)
summary
# Expected qualitative ranking (exact numbers will vary with the route_pairs list and
# machine speed, but the ORDER should be stable):
#   avg_path_cost:      UCS == A* (both optimal, tied for lowest) < Greedy <= BFS < DFS (typically highest)
#   avg_nodes_expanded: Greedy and A* lowest; UCS and BFS/DFS higher, with UCS often highest
#                       since it has no directional guidance at all.


In [ ]:
def plot_summary_metric(summary, metric, title, ylabel):
    names = list(summary.keys())
    values = [summary[name][metric] for name in names]

    plt.figure(figsize=(9, 4))
    plt.bar(names, values)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()


plot_summary_metric(summary, "avg_nodes_expanded", "Average Nodes Expanded by Algorithm", "Average nodes expanded")
plot_summary_metric(summary, "avg_path_cost", "Average Path Cost by Algorithm", "Average path cost (km)")


### Checkpoint 5 -- Instructor Answer Key

1. **Lowest average path cost:** UCS and A* should tie for the lowest average path cost, since both are guaranteed optimal (UCS always; A* because `straight_line_distance` is admissible). This matches AIMA's stated guarantees, so it should not surprise students who did Checkpoints 2-3 carefully.
2. **Fewest nodes expanded / cost tradeoff:** Greedy typically expands the fewest nodes on average, but it does *not* have the lowest path cost -- it trades solution quality for speed. This is the central informed-vs-optimal tradeoff AIMA discusses: greedy is fast but not optimal, A* is nearly as fast (when the heuristic is good) *and* optimal, UCS is optimal but slower because it has no heuristic guidance.
3. **DFS vs. A*:** DFS will sometimes accidentally find a cheap path (if it happens to explore a good branch first) but there is no guarantee, and on at least one of the five routes DFS's path cost should be noticeably higher than A*'s. The takeaway: DFS results cannot be trusted for cost-sensitive applications without independently verifying the path, since it has no cost-awareness built in at all.
4. **Heuristic effect on A* vs. UCS:** The straight-line-distance heuristic lets A* "aim" toward the goal, so it should expand fewer (or equal) nodes than UCS on every route while still returning the same optimal cost -- this is the practical benefit of a good admissible heuristic: same guarantee, less work.
5. **Instant + low-memory scenario:** The expected answer is **greedy best-first search**, trading away the optimality guarantee for speed and a smaller frontier. Accept A* as an answer only if the student explicitly justifies that A*'s cost/memory overhead is still acceptable for their use case; the key grading criterion is that the student correctly identifies *what is being given up* (optimal path cost) in exchange for speed.


## 13. Extension Challenge -- Reference Solutions

These are reference implementations for the three extension options. Students only need to complete **one**; use whichever option matches what a given student attempted.

### Option A: Iterative Deepening Search


In [ ]:
def depth_limited_search(problem, limit):
    """Returns (node, nodes_expanded, cutoff_occurred)."""
    nodes_expanded = 0

    def recursive_dls(node, problem, limit, nodes_expanded):
        if problem.goal_test(node.state):
            return node, nodes_expanded, False
        elif limit == 0:
            return None, nodes_expanded, True
        else:
            cutoff_occurred = False
            for child in node.expand(problem):
                nodes_expanded += 1
                result, nodes_expanded, child_cutoff = recursive_dls(child, problem, limit - 1, nodes_expanded)
                if child_cutoff:
                    cutoff_occurred = True
                elif result is not None:
                    return result, nodes_expanded, False
            return None, nodes_expanded, cutoff_occurred

    return recursive_dls(Node(problem.initial), problem, limit, nodes_expanded)


def iterative_deepening_search(problem, max_depth=50):
    total_expanded = 0
    for depth in range(max_depth):
        result, expanded, cutoff = depth_limited_search(problem, depth)
        total_expanded += expanded
        if result is not None:
            return result, total_expanded
        if not cutoff:
            return None, total_expanded
    return None, total_expanded


ids_problem = GraphProblem("Oradea", "Eforie", romania_map)
ids_solution, ids_expanded = iterative_deepening_search(ids_problem)
bfs_cmp_solution, bfs_cmp_expanded = breadth_first_graph_search(ids_problem)
dfs_cmp_solution, dfs_cmp_expanded = depth_first_graph_search(ids_problem)

print("IDS  path cost:", ids_solution.path_cost, "total nodes expanded (across all iterations):", ids_expanded)
print("BFS  path cost:", bfs_cmp_solution.path_cost, "nodes expanded:", bfs_cmp_expanded)
print("DFS  path cost:", dfs_cmp_solution.path_cost, "nodes expanded:", dfs_cmp_expanded)


**Discussion point for Option A:** IDS re-expands shallow nodes on every iteration, so its *total* nodes-expanded count (summed across all depth limits) is typically higher than a single BFS pass, but IDS uses only O(depth) memory versus BFS's O(branching_factor^depth) -- the same fundamental tradeoff AIMA highlights between IDS and BFS.

### Option B: Weighted A*


In [ ]:
def weighted_astar_search(problem, h, w=1.0):
    return best_first_graph_search(problem, lambda n: n.path_cost + w * h(n))


weighted_problem = GraphProblem("Arad", "Bucharest", romania_map)
weighted_h = lambda n: straight_line_distance(n.state, weighted_problem.goal, romania_map.locations)

for w in [0.0, 1.0, 3.0]:
    solution, expanded = weighted_astar_search(weighted_problem, weighted_h, w=w)
    print(f"w={w}: path_cost={solution.path_cost}, nodes_expanded={expanded}")
# Expected: w=0.0 behaves exactly like UCS (cost 418), w=1.0 matches plain A* (cost 418,
# fewer expansions than w=0.0), w=3.0 typically expands the fewest nodes but may return a
# higher path cost since w>1 breaks the admissibility guarantee.


**Discussion point for Option B:** this is a good moment to make the completeness/optimality/speed tradeoff concrete and adjustable rather than a fixed property of an algorithm -- `w=0` recovers UCS's guarantee exactly, `w=1` recovers A*'s guarantee exactly, and `w>1` sacrifices the guarantee for speed, functioning as a dial between UCS-like and greedy-like behavior.

### Option C: A New 8-Puzzle Heuristic


In [ ]:
def max_heuristic(node):
    return max(misplaced_tiles(node), manhattan_distance(node))


for name, h in [("misplaced_tiles", misplaced_tiles),
                ("manhattan_distance", manhattan_distance),
                ("max_heuristic", max_heuristic)]:
    solution, expanded = astar_search(puzzle, h)
    print(f"{name}: moves={len(solution.solution())}, nodes_expanded={expanded}")
# Expected: max_heuristic expands the fewest (or tied-fewest) nodes, since max() of two
# admissible heuristics is itself admissible and at least as informed as either alone.


**Discussion point for Option C:** `max(h1, h2)` of two admissible heuristics is always admissible (since it still never exceeds the true cost, as both `h1` and `h2` individually don't) and is always at least as informed as either heuristic alone. This is a useful general-purpose technique students can reuse: if you have two cheap admissible heuristics, taking their max is free extra information.


## 14. Final Reflection -- Instructor Notes

**Grading guidance:** look for a paragraph that connects at least three of these ideas with evidence from the student's own experiment output (not just recited definitions):

- **No information (BFS/DFS):** fast to implement, cheap per-node, but no optimality guarantee (DFS) or only a weak one (BFS, optimal only for unit costs).
- **Path cost only (UCS):** optimality guarantee, but the most nodes expanded of the "good" algorithms in this lab, because it has no sense of direction toward the goal.
- **Path cost + heuristic (Greedy/A*):** heuristic information lets the search prune much of the space; A* keeps UCS's optimality guarantee *only if* the heuristic is admissible, while greedy trades that guarantee away entirely for speed.
- **Is more information always worth it?** No -- computing/maintaining a good heuristic has its own cost (design effort, and sometimes computation cost per node), and for very small problems (like this 20-city map) the overhead of an uninformed search may not matter in practice even though it matters asymptotically. The "worth it" answer should reference time/space complexity in the large-scale case, not just this lab's small map.

A strong answer will explicitly distinguish **completeness**, **optimality**, and **efficiency (time/space)** as three separate axes, rather than treating "better" as a single scale.

## Submission Checklist (for grading reference)

- [ ] Notebook runs top to bottom without errors
- [ ] All 13 TODOs completed correctly (see SOLUTION comments above for exact expected code/behavior)
- [ ] All 5 checkpoints answered with reasoning, not just conclusions
- [ ] Experiment summary table and both required plots are present and rendered
- [ ] One extension option attempted (bonus, if your syllabus treats it as such)
- [ ] Final reflection paragraph references specific results from their own run, not generic AIMA definitions
